# Data preparation — PhysioNet Challenge 2017

## Pipeline de referência e divisão federada train/dev/test

O artigo divide o treino original em oito nós de 1.000 registos e junta os 530 registos restantes aos 300 registos do conjunto de validação para construir o test (830 registos). Esta pasta local contém os **8.528 registos rotulados de treino**, mas não contém o conjunto oficial de validação de 300 registos. Por isso, esta preparação mantém os mesmos oito clientes virtuais, mas atribui seis clientes a **train**, um a **dev** e um a **test**. Os oito clientes têm tamanhos semelhantes e distribuição de classes estratificada; dev/test não são independentes por instituição, são partições simuladas desta cópia local.

O pipeline de sinal segue os passos descritos: ECG single-lead a 300 Hz, reamostragem para 200 Hz e comprimento fixo de 30 s (6.000 amostras), com zero-padding no fim dos sinais curtos e truncagem central dos longos. **Como extensão ao artigo**, cada ECG é normalizado individualmente por z-score (média zero e desvio-padrão unitário), calculado sobre as amostras reais após reamostragem e ajuste temporal. Nos sinais curtos, o padding é acrescentado depois da normalização e permanece zero. Não é aplicada filtragem manual adicional; `resample_poly` usa o filtro anti-aliasing necessário à reamostragem. As labels originais são mantidas e também codificadas como N=0, A=1, O=2, ~=3.

A execução cria um `.npz` por cliente, manifestos CSV e um JSON com a configuração. A normalização por registo não usa estatísticas globais aprendidas a partir dos conjuntos de treino, dev ou teste; por isso, é aplicada de forma independente a cada sinal.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import resample_poly

N_CLIENTS = 8
N_TRAIN_CLIENTS = 6
SOURCE_FS = 300
TARGET_FS = 200
DURATION_SECONDS = 30
TARGET_LENGTH = TARGET_FS * DURATION_SECONDS
SEED = 2024
LABEL_TO_ID = {"N": 0, "A": 1, "O": 2, "~": 3}

# Localiza a raiz do projeto tanto a partir da raiz como da pasta DataPreparation.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "datasets" / "training2017" / "REFERENCE.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "datasets" / "training2017"
REFERENCE = DATA_DIR / "REFERENCE.csv"
if not REFERENCE.is_file():
    raise FileNotFoundError(f"Não encontrei o ficheiro de labels: {REFERENCE}")

labels = pd.read_csv(REFERENCE, header=None, names=["record", "label"])
labels["record"] = labels["record"].astype(str).str.strip()
labels["label"] = labels["label"].astype(str).str.strip()
if labels["record"].duplicated().any():
    raise ValueError("REFERENCE.csv tem IDs duplicados.")
unknown_labels = set(labels["label"]) - set(LABEL_TO_ID)
if unknown_labels:
    raise ValueError(f"Labels não reconhecidas: {sorted(unknown_labels)}")

missing = [
    record for record in labels["record"]
    if not (DATA_DIR / f"{record}.mat").is_file()
    or not (DATA_DIR / f"{record}.hea").is_file()
]
if missing:
    raise FileNotFoundError(f"Faltam ficheiros .mat/.hea para {len(missing)} registos; ex.: {missing[:5]}")
if len(labels) < N_CLIENTS:
    raise ValueError(f"São necessários pelo menos {N_CLIENTS} registos.")

# Distribui cada classe pelos oito clientes, de forma reproduzível e quase IID.
rng = np.random.default_rng(SEED)
client_frames = []
for label in sorted(LABEL_TO_ID):
    records = labels.loc[labels["label"] == label, "record"].to_numpy().copy()
    rng.shuffle(records)
    for client_index, chunk in enumerate(np.array_split(records, N_CLIENTS), start=1):
        client_frames.append(pd.DataFrame({"record": chunk, "label": label, "client": f"client_{client_index:02d}"}))
manifest = pd.concat(client_frames, ignore_index=True)
client_roles = {
    **{f"client_{i:02d}": "train" for i in range(1, N_TRAIN_CLIENTS + 1)},
    f"client_{N_TRAIN_CLIENTS + 1:02d}": "dev",
    f"client_{N_TRAIN_CLIENTS + 2:02d}": "test",
}
manifest["split"] = manifest["client"].map(client_roles)
manifest["label_id"] = manifest["label"].map(LABEL_TO_ID).astype(np.int64)
manifest["mat_path"] = manifest["record"].map(lambda r: f"datasets/training2017/{r}.mat")
assert len(manifest) == len(labels) == 8528
assert manifest["record"].is_unique
assert manifest["split"].notna().all()

def read_header(record):
    """Lê frequência, número de derivações e amostras do header WFDB."""
    header = (DATA_DIR / f"{record}.hea").read_text().splitlines()
    fields = header[0].split()
    return int(fields[1]), int(float(fields[2])), int(fields[3])

def preprocess_record(record):
    """Reamostra para 200 Hz e fixa em 30 s, sem normalizar a amplitude."""
    n_leads, fs, n_header_samples = read_header(record)
    if n_leads != 1:
        raise ValueError(f"Esperava ECG single-lead; {record} tem {n_leads} derivações.")
    if fs != SOURCE_FS:
        raise ValueError(f"Frequência inesperada em {record}: {fs} Hz (esperado {SOURCE_FS} Hz).")
    values = np.asarray(loadmat(DATA_DIR / f"{record}.mat")["val"]).squeeze()
    if values.ndim != 1 or values.size != n_header_samples:
        raise ValueError(f"Dimensão/comprimento do sinal não corresponde ao header: {record}.")
    if not np.isfinite(values).all():
        raise ValueError(f"O sinal contém NaN ou infinito: {record}.")

    # 300 -> 200 Hz equivale à razão racional 2/3.
    signal = resample_poly(values.astype(np.float32), up=2, down=3)
    if signal.size > TARGET_LENGTH:
        # Mantém a janela central de 30 s, como descrito no artigo.
        start = (signal.size - TARGET_LENGTH) // 2
        signal = signal[start:start + TARGET_LENGTH]

    # Extensão ao artigo: normaliza cada ECG individualmente antes do padding.
    signal = signal.astype(np.float64, copy=False)
    signal_mean = signal.mean()
    signal_std = signal.std()
    if not np.isfinite(signal_std) or signal_std <= np.finfo(np.float64).eps:
        raise ValueError(f"Desvio-padrão inválido para normalizar o registo {record}.")
    signal = (signal - signal_mean) / signal_std

    if signal.size < TARGET_LENGTH:
        # O padding fica a zero e não altera as estatísticas do ECG real.
        signal = np.pad(signal, (0, TARGET_LENGTH - signal.size), mode="constant")
    if signal.size != TARGET_LENGTH:
        raise RuntimeError(f"Comprimento final inesperado em {record}: {signal.size}.")
    return signal.astype(np.float32, copy=False)

OUTPUT_DIR = PROJECT_ROOT / "DataPreparation" / "physionet2017_prepared_normalized"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest["processed_path"] = manifest["client"].map(lambda c: f"{c}.npz")
manifest.to_csv(OUTPUT_DIR / "manifest.csv", index=False)

# Cada cliente obtém o seu próprio array preparado; dev/test não entram no treino.
for client in sorted(client_roles):
    client_data = manifest.loc[manifest["client"] == client].reset_index(drop=True)
    X = np.stack([preprocess_record(record) for record in client_data["record"]])
    y = client_data["label_id"].to_numpy(dtype=np.int64)
    np.savez_compressed(
        OUTPUT_DIR / f"{client}.npz",
        X=X,
        y=y,
        record_ids=client_data["record"].to_numpy(dtype=str),
        labels=client_data["label"].to_numpy(dtype=str),
    )
    client_data.to_csv(OUTPUT_DIR / f"{client}.csv", index=False)

for split in ("train", "dev", "test"):
    manifest.loc[manifest["split"] == split].to_csv(OUTPUT_DIR / f"{split}.csv", index=False)

config = {
    "source_sampling_rate_hz": SOURCE_FS,
    "target_sampling_rate_hz": TARGET_FS,
    "resampling_ratio": "2/3",
    "duration_seconds": DURATION_SECONDS,
    "samples_per_record": TARGET_LENGTH,
    "short_signal_handling": "zero-pad on right",
    "long_signal_handling": "keep central 30 seconds",
    "amplitude_normalization": "per-record z-score (mean 0, standard deviation 1)",
    "normalization_order": "after resampling and temporal crop, before zero-padding",
    "normalization_scope": "real ECG samples within each record; no cross-record statistics",
    "manual_filtering": "none; anti-aliasing filter inherent to resample_poly",
    "label_to_id": LABEL_TO_ID,
    "seed": SEED,
    "client_roles": client_roles,
    "source_validation_set_available": False,
}
with (OUTPUT_DIR / "preprocessing_config.json").open("w") as f:
    json.dump(config, f, indent=2)

# Validar tamanhos, classes e a forma/tipo dos arrays gerados.
client_summary = manifest.pivot_table(
    index=["client", "split"], columns="label", values="record",
    aggfunc="count", fill_value=0,
).reindex(columns=sorted(LABEL_TO_ID), fill_value=0)
client_summary["total"] = client_summary.sum(axis=1)
display(client_summary)
split_summary = manifest.groupby("split").agg(
    clients=("client", "nunique"), records=("record", "size")
).reindex(["train", "dev", "test"])
display(split_summary)

sample_client = sorted(client_roles)[0]
with np.load(OUTPUT_DIR / f"{sample_client}.npz") as sample:
    assert sample["X"].shape[1] == TARGET_LENGTH
    assert sample["X"].dtype == np.float32
    assert np.isfinite(sample["X"]).all()
    print("Array de exemplo:", sample_client, sample["X"].shape, sample["y"].shape)
print(f"\nDados preparados e manifestos guardados em: {OUTPUT_DIR}")